In [2]:
import io
import logging
import sys
from pathlib import Path

import boto3
import pandas as pd
from botocore.exceptions import BotoCoreError, ClientError
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (
        path
        for path in (working_directory, *working_directory.parents)
        if (path / "src" / "modelo" / "modelo.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.common.storage import get_storage_settings, s3_uri_for
from src.modelo.modelo import (
    MUNICIPALITY_CANDIDATES_FILENAME,
    RESULTS_DIR,
    RESULTS_PREFIX,
)
from src.negocio.supuestos import PESOS_CENTRAL, SUPUESTOS, parametros

In [3]:
storage_settings = get_storage_settings()
candidatos_path = RESULTS_DIR / MUNICIPALITY_CANDIDATES_FILENAME
candidatos_s3_uri = (
    s3_uri_for(MUNICIPALITY_CANDIDATES_FILENAME, RESULTS_PREFIX)
    if storage_settings.save_s3
    else None
)

if candidatos_s3_uri:
    bucket_and_key = candidatos_s3_uri.removeprefix("s3://").split("/", 1)
    client_options = (
        {"region_name": storage_settings.region_name}
        if storage_settings.region_name
        else {}
    )
    try:
        response = boto3.client("s3", **client_options).get_object(
            Bucket=bucket_and_key[0],
            Key=bucket_and_key[1],
        )
        with response["Body"] as body:
            municipios_candidatos = pd.read_parquet(io.BytesIO(body.read()))
        print(f"Candidatos cargados desde {candidatos_s3_uri}")
    except (BotoCoreError, ClientError) as error:
        if not candidatos_path.is_file():
            raise RuntimeError(
                f"No se pudo leer {candidatos_s3_uri} y no existe copia local en "
                f"{candidatos_path}."
            ) from error
        logging.getLogger(__name__).warning(
            "No se pudo leer candidatos desde S3; se usará la copia local: %s",
            error,
        )
        municipios_candidatos = pd.read_parquet(candidatos_path)
else:
    if not candidatos_path.is_file():
        raise FileNotFoundError(
            f"No se encontró el resultado local de candidatos: {candidatos_path}; "
            "ejecuta primero la selección de municipios en 03_modelo.ipynb."
        )
    municipios_candidatos = pd.read_parquet(candidatos_path)
    print(f"Candidatos cargados desde {candidatos_path}")

print(f"Municipios candidatos: {len(municipios_candidatos)}")
display(municipios_candidatos)

Candidatos cargados desde C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\results\municipios_candidatos.parquet
Municipios candidatos: 562


,municipality_code,municipality_name,province_code,province,population_year,population_total,population_18_50,population_growth_start_year,population_growth_pct,income_year,mean_net_income_per_person,gyms_count,other_sports_facilities_count,cluster,tasa_ponderada_de_competidores,media_tasa_ponderada_cluster
0,04001,Abla,04,Almería,2025,1267,464,2021,0.555556,2024,12836,0,1,1,7.892660,28.792748
1,04002,Abrucena,04,Almería,2025,1235,442,2021,1.89769,2024,12565,0,0,1,0.000000,28.792748
2,04004,Albanchez,04,Almería,2025,684,174,2021,-5.0,2024,12885,0,0,1,0.000000,28.792748
3,04005,Alboloduy,04,Almería,2025,578,216,2021,-5.863192,2024,11848,0,0,1,0.000000,28.792748
4,04007,Alcolea,04,Almería,2025,831,285,2021,-2.579132,2024,11594,0,1,1,12.033694,28.792748
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557,41099,Villanueva del Río y Minas,41,Sevilla,2025,5079,2074,2021,3.273688,2024,10788,0,1,2,1.968892,12.482330
558,41100,Villanueva de San Juan,41,Sevilla,2025,980,351,2021,-8.837209,2024,11249,0,0,1,0.000000,28.792748
559,41902,Isla Mayor,41,Sevilla,2025,5760,2492,2021,-1.352971,2024,10100,0,6,2,10.416667,12.482330
560,41903,"Cuervo de Sevilla, El",41,Sevilla,2025,8779,3882,2021,1.526541,2024,10378,0,4,2,4.556328,12.482330


## Supuestos del negocio
Aunque contemplamos en supuestos 3 tipos de supuestos nos quedamos con el escenario central. 

In [4]:
parametros_centrales = parametros('central')
claves_supuestos = [
    'penetracion', 'ticket_mensual', 'capacidad', 'cuota_max',
    'atractivo_nuevo', 'coste_anual_club',
]
supuestos_view = pd.DataFrame([
    {
        'Supuesto': clave,
        'Valor central': parametros_centrales[clave],
        'Fuente / criterio': SUPUESTOS[clave]['fuente'],
        'URL': SUPUESTOS[clave]['url'],
        'Consultado': SUPUESTOS[clave]['consultado'],
    }
    for clave in claves_supuestos
])
display(supuestos_view)

,Supuesto,Valor central,Fuente / criterio,URL,Consultado
0,penetracion,0.218,Bajo: Informe anual sector fitness España 2025...,https://www.educacionfpydeportes.gob.es/mc/dep...,2026-10-20
1,ticket_mensual,22.370,Bajo: Planet Fitness Clásico 18 €/mes con IVA....,https://corporate.basic-fit.com/investors/annu...,2026-10-20
2,capacidad,2600.000,Socios por centro de los 4 grandes low-cost en...,https://www.gestionfit.com/industria/los-gigan...,2026-10-20
3,cuota_max,0.400,Supuesto del equipo. Sin fuente directa; se pr...,NaN,NaN
4,atractivo_nuevo,2.180,Socios por centro low-cost en España (GestiónF...,https://www.gestionfit.com/industria/los-gigan...,2026-10-20
5,coste_anual_club,556000.000,Basic-Fit FY2025: costes de club (personal 261...,https://annualreport.basic-fit.com/2025/mbr/bu...,2026-10-20


#### Calculo de la demanda potencial (socios potenciales)
demanda_potencial = poblacion_18_50 * penetracion
#### Calculo del mercado potencial
mercado_potencial = demanda_potencial * precio_ticket * 12 meses
#### Demanda por competidores
demanda_por_comp = socios_potenciales / max (competidores_ponderados, 1)

#### cuota centro
cuota_centro = min (f/(competidores_pond + f),cuota_max)
#### socios captables
socios_captables = demanda_potencial * cuota_centro
#### socios_centro
socios_centro = min(socios_captables, capacidad)

In [9]:
metricas = municipios_candidatos.copy()
metricas["competidores_ponderados"] = (
    metricas["gyms_count"]
    + 0.1 * metricas["other_sports_facilities_count"]
)
metricas["socios_potenciales"] = (
    metricas["population_18_50"] * parametros_centrales["penetracion"]
)
metricas["mercado_potencial_eur"] = (
    metricas["socios_potenciales"]
    * parametros_centrales["ticket_mensual"]
    * 12
)
metricas["demanda_por_comp"] = (
    metricas["socios_potenciales"]
    / metricas["competidores_ponderados"].clip(lower=1)
)
atractivo_nuevo = parametros_centrales["atractivo_nuevo"]
metricas["cuota_centro"] = (
    atractivo_nuevo
    / (metricas["competidores_ponderados"] + atractivo_nuevo)
).clip(upper=parametros_centrales["cuota_max"])
metricas["socios_captables"] = (
    metricas["socios_potenciales"] * metricas["cuota_centro"]
)
metricas["socios_centro"] = metricas["socios_captables"].clip(
    upper=parametros_centrales["capacidad"]
)
metricas["facturacion_anual"] = (
    metricas["socios_centro"]
    * parametros_centrales["ticket_mensual"]
    * 12
)
columnas_metricas = [
    "municipality_name", "socios_potenciales", "mercado_potencial_eur",
    "competidores_ponderados", "demanda_por_comp", "cuota_centro",
    "socios_captables", "socios_centro", "facturacion_anual",
]
display(metricas[columnas_metricas])

,municipality_name,socios_potenciales,mercado_potencial_eur,competidores_ponderados,demanda_por_comp,cuota_centro,socios_captables,socios_centro
0,Abla,101.152,27153.24288,0.1,101.152,0.4,40.4608,40.4608
1,Abrucena,96.356,25865.80464,0.0,96.356,0.4,38.5424,38.5424
2,Albanchez,37.932,10182.46608,0.0,37.932,0.4,15.1728,15.1728
3,Alboloduy,47.088,12640.30272,0.0,47.088,0.4,18.8352,18.8352
4,Alcolea,62.130,16678.17720,0.1,62.130,0.4,24.8520,24.8520
...,...,...,...,...,...,...,...,...
557,Villanueva del Río y Minas,452.132,121370.31408,0.1,452.132,0.4,180.8528,180.8528
558,Villanueva de San Juan,76.518,20540.49192,0.0,76.518,0.4,30.6072,30.6072
559,Isla Mayor,543.256,145831.64064,0.6,543.256,0.4,217.3024,217.3024
560,"Cuervo de Sevilla, El",846.276,227174.32944,0.4,846.276,0.4,338.5104,338.5104


# Facturacion

socios_centro × ticket_mensual × 12

In [11]:
## Seleccionamos los 5 municipios con mayor facturación anual"))
top_5_facturacion = metricas.nlargest(5, "facturacion_anual").reset_index(drop=True)
top_5_facturacion.insert(0, "posicion", range(1, len(top_5_facturacion) + 1))
display(top_5_facturacion[[
    "posicion", "municipality_name", "facturacion_anual", "socios_centro",
    "socios_captables", "cuota_centro",
]])

KeyError: 'facturacion_anual'